# Sistema Multi-Agente de Soporte Interno — NubeSoft S.A.

**Proyecto Integrador M3 · Multi-agent orchestration con LangChain + Langfuse**

NubeSoft es una empresa SaaS (ficticia) cuyo equipo de soporte está desbordado porque los tickets
se enrutan mal: preguntas de HR terminan en IT, consultas de Finanzas llegan a Legal, etc.
Este notebook construye un sistema que:

1. **Clasifica la intención** de cada consulta con un **Agente Orquestador** (salida estructurada).
2. **Enruta condicionalmente** la consulta al **Agente RAG especializado** correcto
   (HR, IT/Tech, Finance, Legal) o la rechaza / pide aclaración.
3. Cada agente responde **fundamentado en la documentación interna** de su dominio (FAISS + MMR).
4. Todo el flujo queda **trazado en Langfuse** (clasificador → rama → retriever → generación).
5. **Bonus:** un **Agente Evaluador** puntúa cada respuesta de 1 a 10 y registra los scores en Langfuse.

```mermaid
flowchart LR
    Q[Consulta] --> C[Orquestador<br/>intent_classifier<br/>structured output]
    C --> B{RunnableBranch}
    B -- confianza baja --> CL[Pedir aclaración]
    B -- hr --> HR[HR RAG agent]
    B -- tech --> TE[Tech RAG agent]
    B -- finance --> FI[Finance RAG agent]
    B -- legal --> LE[Legal RAG agent]
    B -- out_of_scope --> OS[Respuesta fuera de alcance]
    HR & TE & FI & LE --> EV[Evaluator<br/>LLM-as-judge]
    EV --> LF[(Langfuse<br/>traces + scores)]
```

**Orden de ejecución:** ejecutar las celdas de arriba hacia abajo (`Run All`). La implementación vive en
`src/`; este notebook la importa, la explica y la ejecuta sección por sección.

## 1. Setup e imports

Cargamos las variables de entorno desde `.env` (ver `.env.example`) y verificamos que estén las claves de
OpenAI y Langfuse. Si Langfuse no está configurado, el sistema funciona igual pero sin tracing.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.config import DOMAINS, langfuse_configured, openai_configured

pd.set_option("display.max_colwidth", 120)

print("OpenAI configurado:  ", openai_configured())
print("Langfuse configurado:", langfuse_configured())
print("Modelo de chat:      ", config.CHAT_MODEL)
print("Modelo de embeddings:", config.EMBEDDING_MODEL)
assert openai_configured(), "Falta OPENAI_API_KEY en el archivo .env (copiar .env.example a .env)"

In [ ]:
# Cliente de Langfuse (singleton). auth_check() valida las claves contra el servidor.
langfuse = None
if langfuse_configured():
    from langfuse import get_client

    langfuse = get_client()
    print("Langfuse auth OK:", langfuse.auth_check())
else:
    print("Langfuse no configurado: se ejecuta sin tracing.")

## 2. Carga de documentos y vector stores

Cada carpeta de `data/` es la base de conocimiento de un agente:

| Carpeta | Agente | Contenido |
|---|---|---|
| `data/hr_docs/` | HR | vacaciones, licencias, beneficios, onboarding, desempeño, conducta + FAQ (CSV) |
| `data/tech_docs/` | IT / Tech | VPN, contraseñas y MFA, hardware, mesa de ayuda e incidentes, software y seguridad + FAQ (CSV) |
| `data/finance_docs/` | Finance | gastos y reembolsos, viajes, facturación, compras, presupuesto y tarjetas + tabla de límites (CSV) |
| `data/legal_docs/` | Legal | privacidad y GDPR, contratos y NDA, propiedad intelectual, compliance + FAQ (CSV) |

**Decisiones de RAG**
- **Loaders:** `TextLoader` (UTF-8) para `.md`/`.txt` y `CSVLoader` para `.csv` (cada fila = un documento,
  ideal para FAQs y tablas de límites). `PyPDFLoader` queda soportado para `.pdf`.
- **Chunking:** `RecursiveCharacterTextSplitter.from_tiktoken_encoder` con **200 tokens / 30 de overlap**,
  cortando primero por encabezados Markdown. Las políticas están escritas en secciones cortas y
  autocontenidas: chunks chicos recuperan exactamente la regla consultada sin arrastrar ruido.
- **Embeddings:** `text-embedding-3-small` (barato y de buena calidad multilingüe).
- **Vector store:** **un índice FAISS por dominio**, persistido en `vectorstores/`. Aislar los índices evita
  que un agente recupere políticas de otra área; FAISS es local, gratuito y sin servidor.
  El índice se reconstruye solo si cambian los documentos (hash de archivos + configuración).

In [ ]:
from src.ingest import domain_files, load_domain_documents, split_domain

rows = []
for key in DOMAINS:
    files = domain_files(key)
    chunks = split_domain(key)
    rows.append({
        "dominio": key,
        "archivos": len(files),
        "formatos": ", ".join(sorted({f.suffix for f in files})),
        "documentos": len(load_domain_documents(key)),
        "chunks": len(chunks),
    })
inventory = pd.DataFrame(rows)
display(inventory)

assert (inventory["chunks"] >= config.MIN_CHUNKS_PER_DOMAIN).all(), "Algún dominio tiene menos de 50 chunks"
print(f"OK: todos los dominios tienen al menos {config.MIN_CHUNKS_PER_DOMAIN} chunks")

In [ ]:
# Ejemplo de un chunk con su metadata (source, domain, chunk_id)
sample = split_domain("finance")[12]
print(sample.metadata)
print(sample.page_content)

In [ ]:
from langchain_openai import OpenAIEmbeddings

from src.ingest import build_or_load_vectorstore

embeddings = OpenAIEmbeddings(model=config.EMBEDDING_MODEL)

vectorstores = {}
for key in DOMAINS:
    store, n_chunks = build_or_load_vectorstore(key, embeddings)
    vectorstores[key] = store
    print(f"{key:8s} -> FAISS con {store.index.ntotal} vectores ({n_chunks} chunks)")

In [ ]:
# Prueba de recuperación directa sobre el índice de Tech
for doc, score in vectorstores["tech"].similarity_search_with_score("la VPN no conecta desde un hotel", k=3):
    print(f"[{doc.metadata['source']} · {doc.metadata['chunk_id']}] distancia={score:.3f}")
    print("   ", doc.page_content[:160].replace("\n", " "), "...\n")

## 3. Definición de agentes RAG especializados

Los cuatro agentes comparten la misma estructura, así que hay **una sola fábrica** (`src/agents/rag_agent.py`)
y cada agente (`hr_agent.py`, `tech_agent.py`, `finance_agent.py`, `legal_agent.py`) es una configuración
con instrucciones propias de su dominio.

Cada agente es una cadena **LCEL**:

```python
RunnablePassthrough.assign(docs=itemgetter("question") | retriever)   # 1. recuperar
    .assign(context=format_docs)                                       # 2. armar contexto con [fuente]
    .assign(answer=prompt | llm | StrOutputParser())                   # 3. generar respuesta
    | format_output                                                    # 4. answer + sources + context
```

- **Retriever MMR** (`k=4`, `fetch_k=20`): Maximal Marginal Relevance trae chunks relevantes *y diversos*,
  evitando 4 fragmentos casi iguales de la misma sección.
- **Prompt de grounding:** responder solo con el contexto, citar `[archivo]` y decir explícitamente
  *"No encuentro esa información"* cuando el contexto no alcanza (anti-alucinación).
- **`temperature=0`** para respuestas consistentes y reproducibles.
- Se usa LCEL y no `LLMChain`/`initialize_agent` (deprecados): LCEL es composable, soporta streaming/batch
  y cada paso aparece como un nodo con nombre propio en la traza de Langfuse.

In [ ]:
from langchain_openai import ChatOpenAI

from src.agents.finance_agent import create_finance_agent
from src.agents.hr_agent import create_hr_agent
from src.agents.legal_agent import create_legal_agent
from src.agents.tech_agent import create_tech_agent

llm = ChatOpenAI(model=config.CHAT_MODEL, temperature=0)

retrievers = {
    key: store.as_retriever(
        search_type="mmr",
        search_kwargs={"k": config.RETRIEVER_K, "fetch_k": config.RETRIEVER_FETCH_K},
    )
    for key, store in vectorstores.items()
}

agents = {
    "hr": create_hr_agent(retrievers["hr"], llm),
    "tech": create_tech_agent(retrievers["tech"], llm),
    "finance": create_finance_agent(retrievers["finance"], llm),
    "legal": create_legal_agent(retrievers["legal"], llm),
}
print("Agentes creados:", list(agents))

In [ ]:
# Invocación directa de un agente (sin orquestador)
out = agents["hr"].invoke({"question": "¿Cuánto dura la licencia por paternidad?"})
display(Markdown(out["answer"]))
print("Fuentes:", out["sources"])
print("Chunks recuperados:", out["retrieved_chunks"])

## 4. Orquestador y enrutamiento condicional

El orquestador (`src/agents/orchestrator.py`) tiene dos piezas:

**a) Clasificador de intención con salida estructurada.** `llm.with_structured_output(RouteDecision)` devuelve
un objeto Pydantic validado:

```python
class RouteDecision(BaseModel):
    department: Literal["hr", "tech", "finance", "legal", "out_of_scope"]
    confidence: float  # 0..1
    reasoning: str
```

¿Por qué así y no como en los ejemplos de clase? El router de keywords (L3) falla con sinónimos y typos;
parsear texto libre (`"hr" in respuesta`, L4) es frágil; tool-calling sin argumentos (L3) no da confianza ni
justificación. Con structured output la categoría está **garantizada dentro del `Literal`** y el
`reasoning` queda en la traza para **depurar misclassifications**. El prompt incluye descripciones de cada
área, reglas de desempate (ej. *el recibo de sueldo es de HR, no de Finance*) y ejemplos few-shot de casos ambiguos.

**b) Enrutamiento condicional con `RunnableBranch`:**

| Condición | Rama |
|---|---|
| `confidence < 0.5` (y no es out_of_scope) | respuesta de **aclaración** (no se adivina) |
| `department == "hr" / "tech" / "finance" / "legal"` | agente RAG correspondiente |
| default | respuesta de **fuera de alcance** (sin gastar retrieval ni LLM) |

In [ ]:
from src.agents.orchestrator import build_classifier, build_orchestrator

classifier = build_classifier(llm)

for q in [
    "¿Cuál es el tope de hotel en Madrid?",
    "Hay un error en los descuentos de mi recibo de sueldo",
    "Tengo un problema",
]:
    decision = classifier.invoke({"question": q})
    print(f"{q!r}\n   -> {decision.department} (confianza {decision.confidence:.2f}): {decision.reasoning}\n")

In [ ]:
orchestrator = build_orchestrator(classifier, agents)

result = orchestrator.invoke({"question": "Ya me aprobaron un curso en PeopleHub, ¿cómo pido que me lo reembolsen?"})
print(f"Ruta: {result['route']} · atendido por: {result['handled_by']} · confianza: {result['confidence']:.2f}")
print("Razonamiento:", result["reasoning"])
display(Markdown(result["answer"]))

## 5. Pruebas y ejemplos

`MultiAgentSystem` (`src/multi_agent_system.py`) ensambla **exactamente las mismas piezas** de las secciones
2-4 (los índices FAISS se cargan desde disco, sin volver a calcular embeddings) y agrega el tracing de
Langfuse y el evaluador. Su método `ask()` es el punto de entrada del sistema.

Las consultas de prueba están en `test_queries.json` (20 consultas con la categoría esperada) e incluyen
casos borde: consultas entre dos áreas, en inglés, con typos, de una palabra, fuera de alcance y una
pregunta cuya respuesta **no** está en la documentación (para verificar que el agente no inventa).

In [ ]:
from src.multi_agent_system import MultiAgentSystem

system = MultiAgentSystem()
print("Chunks por dominio:", system.chunk_counts)
print("Tracing con Langfuse:", system.langfuse is not None)


def show(result: dict) -> None:
    display(Markdown(
        f"**Consulta:** {result['question']}  \n"
        f"**Ruta:** `{result['route']}` → `{result['handled_by']}` (confianza {result['confidence']:.2f})  \n"
        f"**Razonamiento del orquestador:** {result['reasoning']}  \n"
        f"**Fuentes:** {', '.join(result['sources']) or '—'}  \n"
        f"**Trace ID:** `{result['trace_id']}`\n\n---\n\n{result['answer']}"
    ))

In [ ]:
# Ejemplos individuales: uno por agente + fuera de alcance
for q in [
    "¿Cuántos días de vacaciones me corresponden si tengo 3 años de antigüedad?",
    "La VPN me dice 'Handshake did not complete' desde el hotel, ¿qué hago?",
    "¿Cuál es el viático diario si viajo a Estados Unidos?",
    "Un cliente nos pidió firmar su propio NDA, ¿lo puedo firmar yo?",
    "¿Quién ganó el partido de Boca ayer?",
]:
    show(system.ask(q, user_id="demo-notebook", tags=["demo"]))

In [ ]:
# Caso borde: la respuesta NO está en la documentación -> el agente debe admitirlo
show(system.ask("¿NubeSoft ofrece seguro de vida para las mascotas de los empleados?", tags=["demo"]))

In [ ]:
# Suite completa de test queries con medición de accuracy de routing
test_queries = json.loads(Path("test_queries.json").read_text(encoding="utf-8"))
print(f"{len(test_queries)} consultas de prueba")

records = []
for tq in test_queries:
    r = system.ask(
        tq["query"],
        expected_route=tq["expected_route"],   # genera el score routing_correct en Langfuse
        session_id="test-suite",
        tags=["test-suite", tq["case_type"]],
    )
    records.append({
        "id": tq["id"],
        "case_type": tq["case_type"],
        "query": tq["query"],
        "expected": tq["expected_route"],
        "predicted": r["route"],
        "handled_by": r["handled_by"],
        "confidence": round(r["confidence"], 2),
        "correct": r["route"] == tq["expected_route"],
        "answer": r["answer"],
        "context": r["context"],
        "trace_id": r["trace_id"],
    })

results_df = pd.DataFrame(records)
accuracy = results_df["correct"].mean()
print(f"Accuracy de routing: {accuracy:.0%} ({results_df['correct'].sum()}/{len(results_df)})")
display(results_df[["id", "case_type", "query", "expected", "predicted", "handled_by", "confidence", "correct"]])

In [ ]:
# Matriz de confusión: filas = categoría esperada, columnas = ruta elegida
display(pd.crosstab(results_df["expected"], results_df["predicted"], margins=True))

# Accuracy por tipo de caso (standard vs. casos borde)
display(results_df.groupby("case_type")["correct"].agg(["mean", "count"]).rename(columns={"mean": "accuracy"}))

misroutes = results_df[~results_df["correct"]]
print("Consultas mal enrutadas:" if len(misroutes) else "Sin consultas mal enrutadas.")
display(misroutes[["query", "expected", "predicted", "confidence", "trace_id"]])

## 6. Integración con Langfuse

Cada llamada a `system.ask()` genera **una traza** en Langfuse con el árbol completo de ejecución:

```
support_request  (agent · input/output de la consulta, user_id, session_id, tags)
└── orchestrator
    ├── intent_classifier        ← prompt, RouteDecision (department, confidence, reasoning), tokens, costo
    ├── conditional_router
    │   └── hr_rag_agent         ← la rama elegida
    │       ├── hr_retriever     ← chunks recuperados con su contenido y fuente
    │       └── hr_generation    ← prompt con contexto + respuesta del LLM
    ├── finalize
    └── quality_evaluator        ← (bonus) juez LLM + scores
Scores: route · routing_confidence · routing_correct · relevance · completeness · accuracy · quality_score · needs_human_review
```

**Cómo se instrumenta** (`MultiAgentSystem.ask`):
- `langfuse.start_as_current_observation(name="support_request", as_type="agent")` crea el span raíz.
- `propagate_attributes(user_id, session_id, tags, trace_name)` propaga los atributos a toda la traza.
- `langfuse.langchain.CallbackHandler` traza **automáticamente** cada Runnable de LangChain (clasificador,
  rama, retriever, LLM) anidado bajo el span raíz, con latencias, tokens y costo.
- `score_trace(...)` registra la ruta elegida, la confianza y, si se conoce la categoría esperada, `routing_correct`.

**Flujo de depuración de una consulta mal enrutada:**
1. En Langfuse → *Traces*, filtrar por score `routing_correct = 0` (o por `routing_confidence` baja).
2. Abrir la traza y mirar el `reasoning` del `intent_classifier`: ¿por qué eligió esa área?
3. Si el routing fue correcto pero la respuesta es mala, abrir el `*_retriever` y revisar qué chunks se
   recuperaron (¿retrieval fallido o generación incorrecta?).
4. Ajustar el prompt del clasificador (reglas / few-shot) o los documentos y volver a correr la suite.

In [ ]:
if system.langfuse is not None:
    print("Ejemplos de trazas en Langfuse:")
    for _, row in results_df.head(5).iterrows():
        print(f"  [{row['predicted']:12s}] {system.langfuse.get_trace_url(trace_id=row['trace_id'])}")
    if len(misroutes):
        print("\nTrazas de consultas mal enrutadas (para depurar):")
        for _, row in misroutes.iterrows():
            print(f"  {row['query'][:60]!r}: {system.langfuse.get_trace_url(trace_id=row['trace_id'])}")
else:
    print("Langfuse no está configurado: completar LANGFUSE_* en .env para ver las trazas.")

In [ ]:
# Conversación agrupada en una sesión: en Langfuse -> Sessions se ve como un único hilo
session = "demo-session-onboarding"
for q in [
    "Entro a trabajar el lunes, ¿qué tengo que completar antes del primer día?",
    "¿Cuándo me llega la notebook y cómo instalo la VPN?",
    "¿Cuánto me dan para equipar mi escritorio en casa?",
]:
    r = system.ask(q, user_id="nuevo-ingreso-42", session_id=session, tags=["demo-session"])
    print(f"[{r['route']:8s}] {q}")

system.flush()  # imprescindible en notebooks: envía los eventos pendientes a Langfuse

## 7. BONUS — Agente Evaluador con la Score API de Langfuse

`src/evaluator.py` implementa un **evaluador LLM-as-a-judge** que recibe la consulta original, el contexto
recuperado y la respuesta final, y devuelve (con structured output) puntajes enteros de **1 a 10**:

| Score en Langfuse | Qué mide |
|---|---|
| `relevance` | ¿responde lo que se preguntó? |
| `completeness` | ¿cubre todas las partes, con plazos, montos, pasos y canales? |
| `accuracy` | ¿cada afirmación está respaldada por el **contexto recuperado**? (detecta alucinaciones) |
| `quality_score` | puntaje global 1-10 (el pedido por la consigna); una respuesta inventada nunca supera 4 |
| `needs_human_review` | `true` si `quality_score < 6`: la respuesta no debería llegar al cliente sin revisión |

El evaluador corre como una observación de tipo `evaluator` **dentro de la misma traza** y registra los
puntajes con `langfuse.create_score(trace_id=..., name=..., value=..., comment=reasoning)`.
También puede evaluar trazas ya existentes pasando `trace_id` (evaluación a posteriori).

Se eligió implementarlo en código (y no solo con los evaluadores configurables desde la UI de Langfuse)
para que sea **reproducible, versionado en el repo y parte del flujo**: así se puede bloquear una respuesta
de baja calidad *antes* de enviarla.

In [ ]:
# Evaluación en línea: ask(..., evaluate=True)
r = system.ask("¿Qué hago si me roban la notebook del trabajo?", evaluate=True, tags=["evaluated"])
show(r)
pd.DataFrame([r["evaluation"]])

In [ ]:
# Evaluación a posteriori de todas las respuestas RAG de la suite de pruebas (se adjunta a cada traza)
rag_rows = results_df[results_df["handled_by"].str.endswith("_agent")]
evaluations = []
for _, row in rag_rows.iterrows():
    ev = system.evaluator.evaluate_and_score(
        row["query"], row["answer"], row["context"], trace_id=row["trace_id"]
    )
    evaluations.append({"id": row["id"], "query": row["query"], **{k: v for k, v in ev.items() if k != "trace_id"}})

eval_df = pd.DataFrame(evaluations)
display(eval_df)
print(eval_df[["relevance", "completeness", "accuracy", "quality_score"]].mean().round(2))
print("Respuestas marcadas para revisión humana:", int(eval_df["needs_human_review"].sum()))

In [ ]:
# Falla inyectada: una respuesta deliberadamente inventada debe recibir un puntaje bajo.
question = "¿Cuántos días de vacaciones tengo con 3 años de antigüedad?"
good = system.ask(question)
hallucinated = (
    "Con 3 años de antigüedad te corresponden 45 días corridos de vacaciones, que podés "
    "cobrar en efectivo si no los usás. [01_vacaciones_y_licencias.md]"
)

comparison = pd.DataFrame([
    {"respuesta": "agente RAG", **system.evaluator.evaluate(question, good["answer"], good["context"]).model_dump()},
    {"respuesta": "inventada", **system.evaluator.evaluate(question, hallucinated, good["context"]).model_dump()},
])
display(comparison)
assert comparison.loc[1, "overall_score"] < comparison.loc[0, "overall_score"], "El juez no detectó la alucinación"

In [ ]:
system.flush()
print("Listo. Revisá en Langfuse: Traces (árbol de ejecución), Scores (calidad y routing) y Sessions.")

## 8. Conclusiones

- El orquestador con structured output enruta correctamente los casos estándar y la mayoría de los casos
  borde; la confianza baja dispara una aclaración en vez de un enrutamiento erróneo.
- Los agentes RAG responden citando la documentación y admiten cuando la información no está.
- Langfuse permite inspeccionar cada paso (clasificación, retrieval, generación) y filtrar errores por score.
- El evaluador genera métricas de calidad continuas y marca respuestas para revisión humana.

Las decisiones técnicas detalladas y las limitaciones conocidas están en el `README.md`.